In [124]:
import numpy as np
import pandas as pd   # We import Pandas!
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn import linear_model
import torch
import itertools

import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from pyro.infer import MCMC, NUTS, HMC, SVI, Trace_ELBO
from pyro.optim import Adam, ClippedAdam

# fix random generator seed (for reproducibility of results)
np.random.seed(42)

# matplotlib options
palette = itertools.cycle(sns.color_palette())
plt.style.use('ggplot')
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)

In [125]:
import pandas as pd

df = pd.read_csv('../../data/processed/pd_2018_present.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 997813 entries, 0 to 997812
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Row ID                997813 non-null  int64  
 1   Incident Number       997813 non-null  int64  
 2   Incident Datetime     997813 non-null  object 
 3   Incident Date         997813 non-null  object 
 4   Incident Time         997813 non-null  object 
 5   Incident Year         997813 non-null  int64  
 6   Incident Code         997813 non-null  int64  
 7   Incident Category     996339 non-null  object 
 8   Incident Subcategory  996339 non-null  object 
 9   Police District       997813 non-null  object 
 10  Latitude              942560 non-null  float64
 11  Longitude             942560 non-null  float64
dtypes: float64(2), int64(4), object(6)
memory usage: 91.4+ MB


In [126]:
df["Incident Datetime"].count(), pd.to_datetime(df["Incident Datetime"], format="%Y/%m/%d %H:%M:%S %p").count()

(np.int64(997813), np.int64(997813))

In [127]:
df["Incident Datetime"] = pd.to_datetime(df["Incident Datetime"], format="%Y/%m/%d %H:%M:%S %p")


In [128]:
# Lets create Year, month, date, weekday, hour

df["year"] = df["Incident Datetime"].dt.year
df["month"] = df["Incident Datetime"].dt.month
df["day"] = df["Incident Datetime"].dt.day
df["weekday"] = df["Incident Datetime"].dt.weekday
df["hour"] = df["Incident Datetime"].dt.hour
df = df.rename(columns={"Police District": "district"})

df_sum = (
    df.groupby(["year", "month", "day", "weekday", "hour", "district"])
    .size()
    .reset_index(name="incident_count")
)

grouped.head()

,year,month,day,weekday,hour,district,incident_count
0,2018,1,1,0,1,Bayview,4
1,2018,1,1,0,1,Central,12
2,2018,1,1,0,1,Ingleside,5
3,2018,1,1,0,1,Mission,6
4,2018,1,1,0,1,Northern,12


In [146]:
X_days = df_sum[["year", "month", "day"]]

X_time = pd.concat(
    [pd.get_dummies(df_sum[x], prefix=x, dtype=int) for x in ["hour", "weekday"]],
    axis=1
)

X_dist = pd.get_dummies(df_sum["district"], prefix="district", dtype=int)

X = pd.concat([X_days, X_time, X_dist], axis=1)

y = df_sum["incident_count"]

X = X.to_numpy()
y = y.to_numpy()

# Standardize X
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [147]:
train_perc = 0.66 # percentage of training data
split_point = int(train_perc*len(y))
perm = np.random.permutation(len(y))
ix_train = perm[:split_point]
ix_test = perm[split_point:]
X_train = X[ix_train,:]
X_test = X[ix_test,:]
y_train = y[ix_train]
y_test = y[ix_test]
print("num train: %d" % len(y_train))
print("num test: %d" % len(y_test))

num train: 204615
num test: 105408


In [136]:
def poisson_model(X, y=None):
    n, p = X.shape

    beta = pyro.sample(
        "beta",
        dist.Normal(torch.zeros(p), torch.ones(p)).to_event(1)
    )
    intercept = pyro.sample("intercept", dist.Normal(0., 1.))

    log_rate = intercept + X @ beta
    rate = torch.exp(log_rate)

    with pyro.plate("data", n):
        pyro.sample("obs", dist.Poisson(rate), obs=y)

In [145]:
print("X_train_torch has NaN:", torch.isnan(X_train_torch).any())
print("y_train_torch has NaN:", torch.isnan(y_train_torch).any())
print("y_train_torch min:", y_train_torch.min(), "max:", y_train_torch.max())
print("X_train_torch shape:", X_train_torch.shape)
print("y_train_torch shape:", y_train_torch.shape)

X_train_torch has NaN: tensor(False)
y_train_torch has NaN: tensor(False)
y_train_torch min: tensor(1.) max: tensor(52.)
X_train_torch shape: torch.Size([204615, 33])
y_train_torch shape: torch.Size([204615])


In [148]:
X_train_torch = torch.tensor(X_train, dtype=torch.float32)
X_test_torch = torch.tensor(X_test, dtype=torch.float32)

y_train_torch = torch.tensor(y_train, dtype=torch.float32)
y_test_torch = torch.tensor(y_test, dtype=torch.float32)

In [149]:
# Define guide function
guide = AutoDiagonalNormal(poisson_model)

# Reset parameter values
pyro.clear_param_store()

In [150]:
# Define the number of optimization steps
n_steps = 1000

# Setup the optimizer
adam_params = {"lr": 0.001}
optimizer = ClippedAdam(adam_params)

# Setup the inference algorithm
elbo = Trace_ELBO(num_particles=1)
svi = SVI(poisson_model, guide, optimizer, loss=elbo)

# Do gradient steps
for step in range(n_steps):
    elbo = svi.step(X_train_torch, y_train_torch)
    if step % 100 == 0:
        print("[%d] ELBO: %.1f" % (step, elbo))

[0] ELBO: 2185651.2
[100] ELBO: 1548235.1
[200] ELBO: 1121785.6
[300] ELBO: 974655.7
[400] ELBO: 766800.1
[500] ELBO: 776613.4
[600] ELBO: 607610.2
[700] ELBO: 594472.8
[800] ELBO: 578164.3
[900] ELBO: 524747.0


Extraxt the posterior and make predictons

In [155]:
from pyro.infer import Predictive

predictive = Predictive(poisson_model, guide=guide, num_samples=1000,
                        return_sites=("intercept", "beta"))
samples = predictive(X_train_torch, y_train_torch)

intercept_samples = samples["intercept"].detach().numpy().squeeze()
beta_samples = samples["beta"].detach().numpy().squeeze()

print("intercept_samples shape:", intercept_samples.shape)
print("beta_samples shape:", beta_samples.shape)
print("X_test shape:", X_test.shape)

# Compute predictions
log_rate_pred = X_test @ beta_samples.T + intercept_samples[None, :]  # (105408, 1000)
rate_pred = np.exp(log_rate_pred)
y_hat = np.mean(rate_pred, axis=1)  # (105408,)

# Since y is not transformed
preds = y_hat
y_true = y_test

# Define compute_error if not already
def compute_error(y_true, y_pred):
    corr = np.corrcoef(y_true, y_pred)[0,1]
    mae = np.mean(np.abs(y_true - y_pred))
    rae = np.mean(np.abs(y_true - y_pred)) / np.mean(np.abs(y_true - np.mean(y_true)))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    r2 = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    return corr, mae, rae, rmse, r2

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

intercept_samples shape: (1000,)
beta_samples shape: (1000, 33)
X_test shape: (105408, 33)
CorrCoef: 0.398
MAE: 1.633
RMSE: 2.362
R2: -0.016


In [ ]:
alpha_samples = samples["alpha"].detach().numpy()
beta_samples = samples["beta"].detach().numpy()
y_hat = np.mean(np.exp(alpha_samples.T + np.dot(X_test, beta_samples[:,0].T)), axis=1)

# convert back to the original scale
preds = y_hat # no need to do any conversion here because the Poisson model received untransformed y's
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))